# Playlist Matcher Test Notebook

This notebook creates a mock music library with the first 10 songs from Favourites.m3u8 and tests the playlist_matcher.py script.

## Setup

First, we'll install the required dependencies and import necessary modules.

In [ ]:
# Install mutagen if not already installed
!pip install mutagen

In [ ]:
import os
import shutil
from pathlib import Path
from mutagen.flac import FLAC
import re
import struct

## Helper Functions

Functions to sanitize filenames and handle special characters.

In [ ]:
def sanitize_filename(name):
    """Sanitize a string for use in filenames by replacing problematic characters"""
    # Replace forward slash with unicode division slash or underscore
    # This preserves the visual appearance while being filesystem-safe
    name = name.replace('/', '∕')  # Unicode division slash U+2215
    # Could also use: name = name.replace('/', '_')
    
    # Replace other problematic characters
    replacements = {
        '\\': '∖',  # Backslash
        ':': '∶',   # Colon  
        '*': '∗',   # Asterisk
        '?': '？',  # Question mark
        '"': '＂',  # Quote
        '<': '＜',  # Less than
        '>': '＞',  # Greater than
        '|': '｜',  # Pipe
    }
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return name

def sanitize_path_component(name):
    """Sanitize directory/album names"""
    return sanitize_filename(name)

## Parse First 10 Entries from Favourites.m3u8

We'll read the playlist and extract metadata for the first 10 songs.

In [ ]:
def parse_playlist_entries(playlist_path, num_entries=10):
    """Parse first N entries from playlist"""
    entries = []
    
    with open(playlist_path, 'r', encoding='utf-8-sig') as f:
        lines = [line.rstrip('\n\r') for line in f.readlines()]
    
    i = 0
    count = 0
    while i < len(lines) and count < num_entries:
        line = lines[i]
        
        if line.startswith('#EXTINF:'):
            if i + 1 < len(lines):
                extinf_line = line
                path_line = lines[i + 1]
                
                # Parse EXTINF: #EXTINF:duration,Artist - Title
                extinf_match = re.match(r'#EXTINF:(\d+),(.+)', extinf_line)
                if extinf_match:
                    duration = extinf_match.group(1)
                    artist_title = extinf_match.group(2)
                    
                    # Split artist and title
                    if ' - ' in artist_title:
                        artist, title = artist_title.split(' - ', 1)
                    else:
                        artist = ""
                        title = artist_title
                    
                    # Parse path: ..\Artist(s)\Album\CD# - Track# - Artist(s) - Title - Album.ext
                    clean_path = path_line.replace('..\\', '').replace('../', '').replace('\\', '/')
                    path_parts = clean_path.split('/')
                    
                    playlist_artist = path_parts[0] if len(path_parts) > 0 else artist
                    album = path_parts[1] if len(path_parts) > 1 else "Unknown Album"
                    filename = path_parts[2] if len(path_parts) > 2 else "unknown.flac"
                    
                    # Parse filename for additional metadata
                    # Format: CD# - Track# - Artist(s) - Title - Album.ext
                    # Use ' - ' (space-dash-space) as delimiter to allow '-' in names
                    filename_match = re.match(r'^(\d+) - (\d+) - (.+?) - (.+?) - (.+?)\.(\w+)$', filename)
                    
                    if filename_match:
                        disc = filename_match.group(1)
                        track = filename_match.group(2)
                        file_artist = filename_match.group(3)
                        file_title = filename_match.group(4)
                        file_album = filename_match.group(5)
                        ext = filename_match.group(6)
                    else:
                        disc = "1"
                        track = str(count + 1)
                        file_artist = artist
                        file_title = title
                        file_album = album
                        ext = "flac"
                    
                    entries.append({
                        'artist': artist.strip(),
                        'title': title.strip(),
                        'album': album.strip(),
                        'playlist_artist': playlist_artist.strip(),
                        'file_artist': file_artist.strip(),
                        'file_title': file_title.strip(),
                        'file_album': file_album.strip(),
                        'disc': disc,
                        'track': track,
                        'ext': ext,
                        'duration': duration,
                        'original_path': path_line
                    })
                    count += 1
                
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    return entries

# Parse first 10 entries
entries = parse_playlist_entries('Favourites.m3u8', 10)

print(f"Parsed {len(entries)} entries:\n")
for i, entry in enumerate(entries, 1):
    print(f"{i}. {entry['artist']} - {entry['title']}")
    print(f"   Album: {entry['album']}")
    print(f"   Track: {entry['disc']}-{entry['track']}")
    print(f"   File title: {entry['file_title']}")
    print()

## Create Mock Library Using Template

Now we'll copy the template FLAC file and modify its metadata for each song.
We sanitize filenames to handle special characters like '/' safely.

In [ ]:
def create_mock_library(entries, template_path='example.flac', base_dir='test_music_library'):
    """Create mock music library by copying template and modifying metadata"""
    
    if not os.path.exists(template_path):
        print(f"Error: Template file {template_path} not found!")
        return []
    
    # Clean up existing directory
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    
    os.makedirs(base_dir)
    
    created_files = []
    
    for entry in entries:
        # Use artist as album artist for the directory structure
        album_artist = entry['artist']
        album = entry['album']
        
        # Sanitize directory names
        safe_artist = sanitize_path_component(album_artist)
        safe_album = sanitize_path_component(album)
        
        # Create directory structure
        album_dir = Path(base_dir) / safe_artist / safe_album
        album_dir.mkdir(parents=True, exist_ok=True)
        
        # Create filename in library format with sanitized components
        # Format: CD# - Track# - Title - Artist(s) - Album.ext
        safe_title = sanitize_filename(entry['title'])
        safe_artist_name = sanitize_filename(entry['artist'])
        safe_album_name = sanitize_filename(entry['album'])
        
        filename = f"{entry['disc']} - {entry['track']} - {safe_title} - {safe_artist_name} - {safe_album_name}.{entry['ext']}"
        file_path = album_dir / filename
        
        try:
            # Copy template file
            shutil.copy2(template_path, file_path)
            
            # Modify metadata - use ORIGINAL unsanitized values for metadata tags
            audio = FLAC(str(file_path))
            audio['title'] = entry['title']  # Original with /
            audio['artist'] = entry['artist']
            audio['album'] = entry['album']
            audio['albumartist'] = album_artist
            audio['tracknumber'] = entry['track']
            audio['discnumber'] = entry['disc']
            audio.save()
            
            # Verify
            audio = FLAC(str(file_path))
            print(f"✓ Created: {file_path.relative_to(base_dir)}")
            print(f"  Metadata: '{audio.get('title', [''])[0]}' by '{audio.get('artist', [''])[0]}'")
            
            created_files.append(str(file_path))
            
        except Exception as e:
            print(f"✗ Failed to create: {file_path.relative_to(base_dir) if file_path.exists() else filename}")
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
    
    return created_files

# Create the mock library
print("Creating mock music library from template...\n")
created_files = create_mock_library(entries)
print(f"\nCreated {len(created_files)} files in test_music_library/")

## Create Test Playlist

Create a small test playlist with just these 10 songs.

In [ ]:
def create_test_playlist(entries, output_path='test_playlist.m3u8'):
    """Create a test playlist with the parsed entries"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('#EXTM3U\n')
        for entry in entries:
            f.write(f"#EXTINF:{entry['duration']},{entry['artist']} - {entry['title']}\n")
            f.write(f"{entry['original_path']}\n")
    
    print(f"Created test playlist: {output_path}")

create_test_playlist(entries)

## Test the Playlist Matcher Script

Now let's run the playlist matcher script on our test data.

In [ ]:
# Run the playlist matcher
!python3 playlist_matcher.py \
    --playlist test_playlist.m3u8 \
    --music-dir test_music_library \
    --output test_output.m3u8 \
    --log test_unmatched.log \
    --format artist_album

## Verify Results

Let's check the output playlist and unmatched log.

In [ ]:
print("=" * 80)
print("OUTPUT PLAYLIST (test_output.m3u8)")
print("=" * 80)
if os.path.exists('test_output.m3u8'):
    with open('test_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("UNMATCHED LOG (test_unmatched.log)")
print("=" * 80)
if os.path.exists('test_unmatched.log'):
    with open('test_unmatched.log', 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("File not found!")

## Verify File Structure

Let's verify the created library structure.

In [ ]:
print("Test Music Library Structure:\n")
for root, dirs, files in os.walk('test_music_library'):
    level = root.replace('test_music_library', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        print(f"{subindent}{file}")